# Session 1: acids, bases and a titration curve
**A. Real-world problem.** A lab wants to know how strong a bottle of acid is. They add a
base from a tall glass tube (a *burette*) a little at a time. The liquid in the flask first
changes slowly and then, all at once, jumps. That jump tells them the concentration.

**Safety.** Handling real acids and bases needs supervision, goggles and a teacher.
Everything here is a computer simulation.

**Plan.** (1) the pH scale, (2) the pH of a strong acid or base, (3) add base to acid and
draw the titration curve, (4) find the equivalence point.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
KW = 1e-14        # ion product of water at 25 degrees C (mol/L)^2

## Step 1: the pH scale
pH measures how much acid (H+ ions) a liquid has: `pH = -log10([H+])`, where `[H+]` is the
concentration in mol/L. A smaller pH means more acid. pH 7 is neutral (pure water at 25 C),
below 7 is acid, above 7 is basic. Each step of 1 in pH is a factor of 10 in [H+].
Example: 0.1 mol/L of a strong acid has [H+] = 0.1, so pH = 1.

In [ ]:
def ph_from_h(h):
    """pH of a solution with hydrogen-ion concentration h (mol/L)."""
    return -np.log10(h)


def ph_strong_acid(c):
    """pH of c mol/L of a strong acid such as HCl (ideal, c well above 1e-6)."""
    return ph_from_h(c) + 0.0       # + 0.0 turns -0.0 into 0.0


def ph_strong_base(c):
    """pH of c mol/L of a strong base such as NaOH at 25 C (ideal, c well above 1e-6)."""
    return 14.0 + np.log10(c)


def kind(ph):
    if abs(ph - 7.0) < 0.5:
        return "neutral"
    return "acidic" if ph < 7 else "basic"

## Step 2: add base to acid
We put `Va` mL of acid (concentration `Ca`) in a flask and add `Vb` mL of base
(concentration `Cb`) from the burette. Acid and base cancel one another mole for mole:

    moles of acid   = Ca * Va / 1000      (mol; Va in mL)
    moles of base   = Cb * Vb / 1000
    leftover        = (acid moles - base moles) / total volume

If acid is left over, the leftover is [H+]. If base is left over, the leftover is [OH-]
and pH = 14 + log10[OH-]. Exactly at the balance, only water is left: pH 7.

In [ ]:
def strong_ph(Ca, Va, Cb, Vb):
    """pH of the flask after adding Vb mL of strong base (Cb, mol/L) to Va mL of strong acid (Ca)."""
    x = (Ca * Va - Cb * Vb) / (Va + Vb)       # leftover acid (+) or base (-), mol/L
    if x >= 0:
        h = (x + np.sqrt(x * x + 4 * KW)) / 2
    else:
        h = 2 * KW / (-x + np.sqrt(x * x + 4 * KW))
    return float(-np.log10(h))


def equivalence_volume(Ca, Va, Cb):
    """Volume of base (mL) that exactly cancels the acid: Ca * Va = Cb * Vb."""
    return Ca * Va / Cb

## Step 3: the curve and the equivalence point
We compute pH after every small step of base. The steep part is the *equivalence point*,
where the moles of base added equal the moles of acid that were in the flask.

In [ ]:
def titration_curve(Ca, Va, Cb, vmax):
    vb = np.linspace(0.0, vmax, 501)
    return vb, np.array([strong_ph(Ca, Va, Cb, v) for v in vb])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    # Figure 1: the pH scale with examples
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.imshow(np.linspace(0, 14, 256)[None, :], extent=[0, 14, 0, 1], aspect="auto", cmap="RdYlBu_r")
    examples = [("1 mol/L HCl", ph_strong_acid(1.0)), ("0.1 mol/L HCl", ph_strong_acid(0.1)),
                ("0.001 mol/L HCl", ph_strong_acid(1e-3)), ("pure water", 7.0),
                ("0.001 mol/L NaOH", ph_strong_base(1e-3)), ("0.1 mol/L NaOH", ph_strong_base(0.1)),
                ("1 mol/L NaOH", ph_strong_base(1.0))]
    for i, (name, ph) in enumerate(examples):
        ax.axvline(ph, color="k", lw=0.8)
        ax.text(ph, 1.04 + 0.28 * (i % 2), f"{name}\npH {ph:.0f}", ha="center", va="bottom", fontsize=7)
        print(f"{name:18s} pH {ph:5.2f}  {kind(ph)}")
    ax.set_ylim(0, 1.8)
    ax.set_yticks([])
    ax.set_xlim(-0.5, 14.5)
    ax.set_xlabel("pH (no unit)")
    ax.set_xticks(range(0, 15))
    ax.set_title("The pH scale at 25 C (ideal solutions)", pad=8)
    fig.tight_layout()
    fig.savefig(OUT / "ph_scale.png", dpi=120)

    # Figure 2: titration curve, 25 mL of 0.1 M HCl with 0.1 M NaOH
    Ca, Va, Cb = 0.1, 25.0, 0.1
    vb, ph = titration_curve(Ca, Va, Cb, 50.0)
    ve = equivalence_volume(Ca, Va, Cb)
    fig2, ax2 = plt.subplots(figsize=(6.5, 4.2))
    ax2.plot(vb, ph, color="tab:blue", lw=2)
    ax2.axvline(ve, color="gray", ls=":")
    ax2.plot([ve], [strong_ph(Ca, Va, Cb, ve)], "ro")
    ax2.annotate(f"equivalence point\n{ve:.1f} mL, pH {strong_ph(Ca, Va, Cb, ve):.0f}",
                 (ve, 7), (ve + 6, 4), arrowprops=dict(arrowstyle="->"))
    ax2.set_xlabel("volume of base added (mL)")
    ax2.set_ylabel("pH of the flask (no unit)")
    ax2.set_title("25 mL of 0.1 mol/L HCl titrated with 0.1 mol/L NaOH")
    ax2.set_ylim(0, 14)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "titration_curve.png", dpi=120)
    for v in (0, 10, 20, 24, 24.9, 25, 25.1, 26, 30, 40):
        print(f"base added {v:5.1f} mL -> pH {strong_ph(Ca, Va, Cb, v):6.3f}")

## Coding tasks
1. **Easy.** Change the concentration of the base in `lesson.py` to 0.2 mol/L and predict the equivalence volume before you run it. Then check it with `equivalence_volume`.
2. **Stretch.** Write `ph_after(vb)` for 50 mL of 0.05 mol/L HCl titrated with 0.05 mol/L NaOH and print the pH at 0, 25, 49, 50 and 51 mL.
3. **Explain.** Explain in your own words why the curve is almost vertical at the equivalence point. Use the idea of a tiny leftover divided by the total volume.

## Assessment
- **A. Conceptual.** Why does the pH hardly change for the first 20 mL of base and then jump within a fraction of a millilitre near 25 mL?
- **B. Numerical.** What is the pH after 20 mL of 0.1 mol/L NaOH has been added to 25 mL of 0.1 mol/L HCl? Show the units and check that the answer is sensible.
- **C. Graph and simulation interpretation.** On the titration curve, how much does the pH change between 24.9 mL and 25.1 mL, and what does that tell you about reading the burette near the end?
- **D. Design and optimisation challenge.** Design: you have 25 mL of 0.1 mol/L acid and a choice of 0.1 mol/L or 0.2 mol/L base. Which do you choose and why? State the equivalence volume for each and the trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("0.1 mol/L HCl has pH 1", abs(L.ph_strong_acid(0.1) - 1.0) < 1e-9))
    checks.append(("each step of 1 in pH is a factor 10 in [H+]", abs(L.ph_from_h(1e-3) - L.ph_from_h(1e-2) - 1.0) < 1e-9))
    checks.append(("0.001 mol/L NaOH has pH 11", abs(L.ph_strong_base(1e-3) - 11.0) < 1e-9))
    checks.append(("pure water (1e-7 mol/L) is neutral", L.kind(L.ph_from_h(1e-7)) == "neutral"))
    checks.append(("the flask starts at pH 1 (0 mL base)", abs(L.strong_ph(0.1, 25.0, 0.1, 0.0) - 1.0) < 1e-6))
    ve = L.equivalence_volume(0.1, 25.0, 0.1)
    checks.append((f"worked example: 25 mL of 0.1 M HCl needs {ve:.1f} mL of 0.1 M NaOH, and the pH there is 7",
                   abs(ve - 25.0) < 1e-12 and abs(L.strong_ph(0.1, 25.0, 0.1, ve) - 7.0) < 1e-6))
    # hand calculation: 10 mL base -> acid left 1.5 mmol in 35 mL -> [H+] = 0.04286 -> pH 1.368
    checks.append(("hand calculation: after 10 mL base pH = -log10(1.5/35) = 1.368",
                   abs(L.strong_ph(0.1, 25.0, 0.1, 10.0) - (-np.log10(1.5 / 35.0))) < 1e-9))
    checks.append(("30 mL of base: pH is above 11", L.strong_ph(0.1, 25.0, 0.1, 30.0) > 11.0))
    vb, ph = L.titration_curve(0.1, 25.0, 0.1, 50.0)
    checks.append(("adding base never lowers the pH", bool(np.all(np.diff(ph) >= -1e-12))))
    checks.append(("the steepest part is at the equivalence volume", abs(vb[np.argmax(np.diff(ph))] - 25.0) < 0.2))
    checks.append(("a 0.2 M base needs only half the volume", abs(L.equivalence_volume(0.1, 25.0, 0.2) - 12.5) < 1e-12))
    checks.append(("finite at the exact boundary and far past it",
                   bool(np.isfinite(L.strong_ph(0.1, 25.0, 0.1, 25.0)) and np.isfinite(L.strong_ph(0.1, 25.0, 0.5, 200.0)))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
